# Práctica: Transfer Learning y Fine Tuning con CNN (Keras)

Objetivo: comparar una CNN hecha desde cero con un **modelo preentrenado** usando:
- Transfer Learning
- Fine Tuning

Dataset: **clasificación de paisajes** utilizado en la unidad anterior.

## 0. Importación de librerías

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


## 1. Preparación de los datos

In [ ]:

IMG_SIZE = 224

train_dir = "data/train"
test_dir = "data/test"

classes = sorted(os.listdir(train_dir))

print("Clases:", classes)


In [ ]:

def load_dataset(directory):
    
    X = []
    y = []
    
    for label, clase in enumerate(os.listdir(directory)):
        
        class_dir = os.path.join(directory, clase)
        
        for file in os.listdir(class_dir):
            
            path = os.path.join(class_dir, file)
            
            try:
                img = Image.open(path).convert("RGB")
                img = img.resize((IMG_SIZE, IMG_SIZE))
                img = np.array(img)
                
                X.append(img)
                y.append(label)
            except:
                pass
    
    return np.array(X), np.array(y)


X_train, y_train = load_dataset(train_dir)
X_test, y_test = load_dataset(test_dir)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


## Normalización y preprocess del modelo

In [ ]:

X_train = preprocess_input(X_train)
X_test = preprocess_input(X_test)


## Visualización de imágenes

In [ ]:

fig, axes = plt.subplots(3,3, figsize=(6,6))

for i, ax in enumerate(axes.flat):
    ax.imshow((X_train[i] + 1)/2)
    ax.set_title(classes[y_train[i]])
    ax.axis("off")

plt.show()


# 2. Modelo base preentrenado (MobileNetV2)

In [ ]:

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

base_model.summary()


## 3. Transfer Learning

In [ ]:

model_tl = models.Sequential([
    
    base_model,
    
    layers.GlobalAveragePooling2D(),
    
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    
    layers.Dense(len(classes), activation="softmax")
])

model_tl.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_tl.summary()


In [ ]:

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_tl = model_tl.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)


## Evaluación Transfer Learning

In [ ]:

pred_probs = model_tl.predict(X_test)
pred_labels = np.argmax(pred_probs, axis=1)

print(classification_report(y_test, pred_labels, target_names=classes))

cm = confusion_matrix(y_test, pred_labels)
print("Confusion matrix:")
print(cm)


# 4. Fine Tuning

In [ ]:

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False


In [ ]:

model_ft = models.Sequential([
    
    base_model,
    
    layers.GlobalAveragePooling2D(),
    
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    
    layers.Dense(len(classes), activation="softmax")
])

model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_ft.summary()


In [ ]:

history_ft = model_ft.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)


## Evaluación Fine Tuning

In [ ]:

pred_probs_ft = model_ft.predict(X_test)
pred_labels_ft = np.argmax(pred_probs_ft, axis=1)

print(classification_report(y_test, pred_labels_ft, target_names=classes))

cm_ft = confusion_matrix(y_test, pred_labels_ft)
print("Confusion matrix:")
print(cm_ft)


# 5. Comparación de resultados


En general los modelos **preentrenados** suelen superar claramente a una CNN creada desde cero porque:

- Ya han aprendido **features visuales generales** (bordes, texturas, patrones).
- Requieren **menos datos para generalizar bien**.
- El **fine tuning** permite ajustar esas features al dataset específico.

Comparación esperada:

| Modelo | Ventajas | Desventajas |
|------|------|------|
| CNN desde cero | Arquitectura flexible | Necesita muchos datos |
| Transfer Learning | Entrena rápido | No ajusta todas las capas |
| Fine Tuning | Mejor rendimiento | Más coste computacional |

Normalmente el **fine tuning obtiene el mejor resultado**.
